In [ ]:
# Chạy một lần, khi các uploader đã dừng. Cần Kaggle Secret HF_TOKEN có quyền phù hợp.
# Xoá vĩnh viễn blob archive trên HF, gồm các bản trong lịch sử. Giữ bản sao raw nếu cần.
HF_ARCHIVES_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_ARCHIVES_PURGE_LFS = True  # False: chỉ xoá folder trên main, chưa thu hồi blob trong lịch sử.
HF_UPLOAD_ARCHIVE = False


def delete_hf_archives():
    import os
    import subprocess
    import sys
    try:
        from huggingface_hub import HfApi, RepoFile, get_token
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'huggingface_hub>=1.19,<3'], check=True)
        from huggingface_hub import HfApi, RepoFile, get_token

    token = os.environ.get('HF_TOKEN') or get_token()
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            raise RuntimeError('Bật Kaggle Secret HF_TOKEN có quyền ghi vào dataset, '
                               'hoặc đăng nhập bằng hf auth login trên local.') from None
    if not token:
        raise RuntimeError('HF_TOKEN rỗng; chưa thực hiện xoá.')

    api = HfApi(token=token)
    if HF_ARCHIVES_PURGE_LFS and not all(hasattr(api, name) for name in
                                       ('list_lfs_files', 'permanently_delete_lfs_files')):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'huggingface_hub>=1.19,<3'], check=True)
        raise RuntimeError('Đã cập nhật huggingface_hub. Restart kernel rồi chạy lại cell này.')

    class ArchiveCleanupError(RuntimeError):
        pass

    repo = HF_ARCHIVES_REPO_ID
    prefix = 'archives/'
    stage = 'inspect'
    try:
        head = api.repo_info(repo_id=repo, repo_type='dataset', revision='main').sha
        tree = [item for item in api.list_repo_tree(
            repo_id=repo, repo_type='dataset', revision=head, recursive=True)
            if isinstance(item, RepoFile)]
        protected = {item.path: item.blob_id for item in tree if not item.path.startswith(prefix)}
        all_lfs = list(api.list_lfs_files(repo_id=repo, repo_type='dataset')) if HF_ARCHIVES_PURGE_LFS else []
        targets = {item.file_oid: item for item in all_lfs
                   if (item.filename or '').startswith(prefix)}
        protected_oids = {item.lfs.sha256 for item in tree
                          if item.lfs and not item.path.startswith(prefix)}
        protected_oids.update(item.file_oid for item in all_lfs
                              if item.filename and not item.filename.startswith(prefix))
        if protected_oids & targets.keys():
            raise ArchiveCleanupError('Có blob archive dùng chung với file ngoài archives/; '
                               'cell dừng trước khi xoá. Cần kiểm tra các blob này riêng.')

        paths = [item.path for item in tree if item.path.startswith(prefix)]
        total_gib = sum(item.size for item in targets.values()) / 1024**3
        print(f'Dataset: {repo} | archives trên main: {len(paths)} file | '
              f'blob archive cần xoá: {len(targets)} ({total_gib:.2f} GiB)', flush=True)
        if paths:
            stage = 'delete_folder'
            commit = api.delete_folder(
                repo_id=repo, repo_type='dataset', revision='main', path_in_repo='archives',
                parent_commit=head, commit_message='Remove crawl archives; keep corpus and diagnostics')
            print('Đã xoá folder archives trên main:', commit.commit_url, flush=True)
        if targets:
            stage = 'purge_archive_lfs'
            api.permanently_delete_lfs_files(
                repo_id=repo, repo_type='dataset', lfs_files=list(targets.values()), rewrite_history=True)
            print(f'Đã gửi yêu cầu xoá vĩnh viễn {len(targets)} blob archive.', flush=True)

        stage = 'verify'
        after_head = api.repo_info(repo_id=repo, repo_type='dataset', revision='main').sha
        after = [item for item in api.list_repo_tree(
            repo_id=repo, repo_type='dataset', revision=after_head, recursive=True)
            if isinstance(item, RepoFile)]
        if any(item.path.startswith(prefix) for item in after):
            raise ArchiveCleanupError('archives/ vẫn có file; kiểm tra uploader khác rồi chạy lại cell.')
        if protected != {item.path: item.blob_id for item in after if not item.path.startswith(prefix)}:
            raise ArchiveCleanupError('Các file ngoài archives/ thay đổi trong lúc dọn; '
                               'kiểm tra commit, có thể có uploader khác đang chạy.')
        print('Kiểm tra xong: archives/ không còn trên main; các file khác giữ nguyên.', flush=True)
        print('Quota có thể cập nhật chậm. Nếu vẫn lỗi private storage, kiểm tra Storage của tổ chức.')
        if not HF_ARCHIVES_PURGE_LFS:
            print('Chưa xoá blob trong lịch sử vì HF_ARCHIVES_PURGE_LFS = False.')
        return {'repo_id': repo, 'status': 'cleaned', 'removed_paths': len(paths),
                'purged_lfs_objects': len(targets), 'archive_bytes_targeted': sum(t.size for t in targets.values())}
    except ArchiveCleanupError:
        raise
    except Exception as error:
        status = getattr(getattr(error, 'response', None), 'status_code', None)
        raise RuntimeError(f'Dọn archives thất bại tại {stage} (HTTP {status or "?"}, '
                           f'{type(error).__name__}). Kiểm tra mạng, quyền write/admin của token '
                           'và chạy lại; bước xoá đã hoàn tất sẽ không tự phục hồi.') from None


hf_archives_cleanup_result = delete_hf_archives()
